# 5. Floating dynamics and the low-level fit API

**Learning goals:** float a resonance mass, assemble a cached likelihood explicitly, validate its gradient, and search multiple starting points.

Run cells from top to bottom in a fresh Python kernel. Install the package and Jupyter
as explained in [the course guide](TUTORIALS.md). No external data files are needed.
Masses are in GeV, invariants in GeV², and daughter indices start at zero.
The small event counts and grid sizes keep this lesson practical on a CPU; they are
teaching settings, not a demonstrated precision choice for a physics analysis.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    DecayChannel, DecayModel, FitSession, NonResonant,
    Parameter, RealImag, Resonance, generate_toy,
)

## Give each dynamical parameter an owner

This section uses the complete B+ -> pi+ pi- pi+ paper-style model. All numerical conventions, charge-dependent coefficients, and normalization choices follow the benchmark.

In [2]:
from dalitzplotfitter import Minimizer, MultiBackgroundNLL
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
mass = Parameter.dynamics("rho.mass", 0.7753, owner="rho",
                          bounds=(0.73, 0.82), step=0.001)
model = DecayModel(
    channel,
    [Resonance("rho", (0, 1), RealImag(1, 0), mass=mass, width=0.1491, spin=1),
     NonResonant(RealImag(0.55, 0.30), name="NR")],
    normalization_method="square-dalitz", normalization_pair=(0, 1),
    normalization_resolution=100,
)
truth = {"rho.mass": 0.7753}

In [3]:
data = generate_toy(
    model, 2500, parameters=truth, seed=2026,
    method="inverse-transform", inverse_resolution=384, include_momenta=False,
)
print(f"Generated {data.size} unweighted events")
assert np.all(np.asarray(data.weights) == 1)

## Compose the likelihood explicitly

For this signal-only, unit-efficiency example, the cached density is intensity divided
by its integral. `MultiBackgroundNLL` also accepts the no-background case. Check that
parameters used by the minimizer match the fixed/floating split baked into the cache.
Rebuild the cache if that split changes; a separately altered parameter list can otherwise
leave a supposed floating parameter with a structural zero gradient.

In [4]:
cache = model.prepare_cache(data)
parameters = model.parameters
cache.check_parameters(parameters)

def signal_density(values):
    return cache.intensity(values) / cache.normalization(values)

objective = MultiBackgroundNLL(signal_density)
minimizer = Minimizer(objective, parameters)
# Check the explicit assembly against the high-level composition at the same point.
session = FitSession(model, data)
np.testing.assert_allclose(objective(truth), session.objective(truth), rtol=1e-12)
gradient = minimizer.check_gradient({"rho.mass": 0.76}, step_scale=1e-5)
print("Maximum absolute gradient discrepancy:", gradient.max_absolute_error)

## Fit from several starts

This section uses the complete B+ -> pi+ pi- pi+ paper-style model. All numerical conventions, charge-dependent coefficients, and normalization choices follow the benchmark.

In [5]:
scan = minimizer.fit_multistart(n_starts=3, seed=51, simplex=True)
result = scan.best
assert result is not None and result.valid
session.report(result)
print("NLL at truth:", float(objective(truth)))
print("Best fitted NLL:", result.fval)
assert result.fval <= float(objective(truth)) + 1e-4
session.plot_projection(result, "s12", bins=40, projection_size=20000)
plt.show()

## Try it yourself

1. Float the width instead, with a positive lower bound and `owner="rho"`.
2. Check the gradient at two interior parameter points.
3. Increase normalization resolution and compare the mass shift to its fitted uncertainty.

## Continue learning

[Next: CP fits](tutorial_06_joint_cp_fit.ipynb). Reference: [performance and caching](../../docs/performance.md), [lineshapes](../../docs/lineshapes.md).

Return to [the course guide](TUTORIALS.md).